# Silver Layer — MovieLens Transformations

**Purpose:** Read raw Bronze tables and apply cleaning and type transformations to produce the Silver layer.

Silver is the single source of truth for analysts and data scientists.
Data is clean, correctly typed, and deduplicated — but still row-level (no aggregations yet).

| Step | Table | Transformations |
|------|-------|-----------------|
| 1 | Create schema | `workspace.silver` |
| 2 | `bronze_ratings` | Unix timestamp → `rated_at`, add `ingestion_date` |
| 3 | `bronze_movies` | Split genres, extract year, clean title, add `ingestion_date` |
| 4 | Save | `silver_ratings`, `silver_movies` |
| 5 | Verify | Row counts + sample rows |

## Config & Imports

In [ ]:
from pyspark.sql.functions import (
    col,
    count,
    current_date,
    from_unixtime,
    regexp_replace,
    split,
    trim,
    when,
)
from pyspark.sql.types import IntegerType

# Source: Bronze Delta tables created by nb_bronze_ingest
SOURCE_SCHEMA = "workspace.bronze"

# Target: Silver layer — clean, typed, pipeline-ready
TARGET_SCHEMA = "workspace.silver"

## Step 1 — Create Silver Schema

`IF NOT EXISTS` makes this safe to re-run at any time.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}")

print(f"Schema ready: {TARGET_SCHEMA}")

## Step 2 — Transform Ratings

**Problem:** `timestamp` is stored as a Unix integer (seconds since 1970-01-01).
That is useless for date filtering or aggregating by month/year in BI tools.

**Fix:** Convert it to a proper `TimestampType` column called `rated_at`, then drop the raw integer.

In [ ]:
# Read raw ratings from Bronze
bronze_ratings_df = spark.table(f"{SOURCE_SCHEMA}.bronze_ratings")

silver_ratings_df = (
    bronze_ratings_df
    # from_unixtime converts the unix integer (e.g. 1147880044) → "2006-05-17 09:34:04"
    # .cast("timestamp") turns that string into a proper TimestampType column called rated_at
    # from_unixtime handles null input natively — returns null, never throws
    .withColumn("rated_at", from_unixtime(col("timestamp")).cast("timestamp"))
    # Drop the original raw integer — rated_at replaces it
    .drop("timestamp")
    # Record the date this row was written by the pipeline (useful for debugging reruns)
    .withColumn("ingestion_date", current_date())
)

print("silver_ratings schema:")
silver_ratings_df.printSchema()

## Step 3 — Transform Movies

Three problems to fix in `bronze_movies`:

| Column | Bronze (raw) | Silver (clean) |
|--------|-------------|----------------|
| `genres` | `"Action\|Adventure\|Comedy"` | `["Action", "Adventure", "Comedy"]` |
| `release_year` | doesn't exist | `1995` (extracted from title) |
| `title` | `"Toy Story (1995)"` | `"Toy Story"` |

In [ ]:
import re
from pyspark.sql.functions import udf

# Python UDF for year extraction.
# Why a UDF instead of regexp_extract(...).cast("int"):
#   Spark's Catalyst optimizer evaluates cast() across ALL rows before the
#   when() condition can filter empty strings, causing CAST_INVALID_INPUT.
#   A Python UDF runs per-row — Python returns None before any cast is attempted,
#   so empty strings and non-matching titles safely become null.
@udf(returnType=IntegerType())
def extract_year(title):
    if title is None:
        return None
    m = re.search(r'\((\d{4})\)$', title.strip())  # match "(1995)" at end of title
    return int(m.group(1)) if m else None            # None → null in Spark

# Read raw movies from Bronze
bronze_movies_df = spark.table(f"{SOURCE_SCHEMA}.bronze_movies")

silver_movies_df = (
    bronze_movies_df

    # ── genres: pipe-string → array ────────────────────────────────────────
    # split() with the [|] character class splits on literal pipe
    # Result: "Action|Adventure" → ["Action", "Adventure"]
    .withColumn("genres", split(col("genres"), "[|]"))

    # ── release_year: extract 4-digit year from title ──────────────────────
    # UDF uses Python re.search — only calls int() when a match is found
    # e.g. "Toy Story (1995)" → 1995  |  "(no genres listed)" → null
    .withColumn("release_year", extract_year(col("title")))

    # ── title: strip the " (1995)" year suffix ─────────────────────────────
    # regexp_replace removes the pattern, trim cleans up any leftover whitespace
    # e.g. "Toy Story (1995)" → "Toy Story"
    .withColumn("title", trim(regexp_replace(col("title"), " \\(\\d{4}\\)", "")))

    # ── ingestion_date: pipeline audit column ──────────────────────────────
    .withColumn("ingestion_date", current_date())
)

print("silver_movies schema:")
silver_movies_df.printSchema()

## Step 4 — Save as Silver Delta Tables

`mode("overwrite")` keeps the notebook idempotent — safe to re-run without duplicating data.

In [ ]:
print("Writing workspace.silver.silver_ratings ...")
(
    silver_ratings_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{TARGET_SCHEMA}.silver_ratings")
)
print("  Done.")

print("Writing workspace.silver.silver_movies ...")
(
    silver_movies_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{TARGET_SCHEMA}.silver_movies")
)
print("  Done.")

print("\nAll Silver tables written.")

## Step 5 — Verify

We read back from the Delta tables on disk (not the DataFrames in memory)
to confirm the data landed correctly and the transformations look right.

In [ ]:
ratings_count = spark.table(f"{TARGET_SCHEMA}.silver_ratings").count()
movies_count  = spark.table(f"{TARGET_SCHEMA}.silver_movies").count()

print("=" * 50)
print("  Silver transform complete")
print("=" * 50)
print(f"  silver_ratings : {ratings_count:>12,} rows")
print(f"  silver_movies  : {movies_count:>12,} rows")
print("=" * 50)

# Show 5 sample rows from each table to visually confirm the transformations
# truncate=False prints full column values without cutting them off
print("\nSample silver_ratings (5 rows):")
spark.table(f"{TARGET_SCHEMA}.silver_ratings").show(5, truncate=False)

print("Sample silver_movies (5 rows):")
spark.table(f"{TARGET_SCHEMA}.silver_movies").show(5, truncate=False)

## Data Quality Check

Count NULLs in every important column across both Silver tables.

- `release_year` NULLs are **expected** — some titles have no year in parentheses
- All other columns should show **0 NULLs**; any non-zero value signals a data problem

In [ ]:
def null_report(table_name, columns):
    """Read a Silver table from disk and print NULL count for each column."""
    df  = spark.table(table_name)
    row = df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in columns
    ]).collect()[0]

    print(f"  {table_name}")
    print(f"  {'Column':<20}  {'NULLs':>10}  Note")
    print("  " + "-" * 45)
    for c in columns:
        note = "<-- expected (no year in title)" if c == "release_year" else ""
        print(f"  {c:<20}  {row[c]:>10,}  {note}")
    print()

print("=" * 55)
print("  Data Quality Check — Silver Layer")
print("=" * 55 + "\n")

null_report(
    f"{TARGET_SCHEMA}.silver_ratings",
    ["userId", "movieId", "rating", "rated_at", "ingestion_date"],
)

null_report(
    f"{TARGET_SCHEMA}.silver_movies",
    ["movieId", "title", "genres", "release_year", "ingestion_date"],
)

print("=" * 55)